# Visualizar y analizar resultados

## Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

In [ ]:
# ================================
# CARGA DE RESULTADOS
# ================================

# Buscar todos los archivos *_resultados.parquet en el directorio actual
archivos_resultados = [f for f in os.listdir() if f.endswith("_resultados.parquet")]

# Cargar todos los resultados en un solo DataFrame
df_global = pd.DataFrame()

for archivo in archivos_resultados:
    try:
        df = pd.read_parquet(archivo)
        # Extraer nombre base para identificar dataset (ej: "df_mcar_10" o "df")
        nombre_dataset = re.sub(r"_resultados\.parquet$", "", archivo)
        df["Dataset"] = nombre_dataset
        df_global = pd.concat([df_global, df], ignore_index=True)
    except Exception as e:
        print(f"⚠️ Error al leer {archivo}: {e}")

# Mostrar preview
print(f"\n✅ Archivos cargados: {len(archivos_resultados)}")
print(f"📊 Total de combinaciones cargadas: {df_global.shape[0]}")
df_global.head()


In [ ]:
# ================================
# CELDA 2: Visualización global - TOP combinaciones por F1
# ================================

# Comprobar si hay datos cargados
if df_global.empty:
    print("⚠️ No se han encontrado resultados para mostrar.")
else:
    # Ordenar por F1 descendente
    df_global = df_global.sort_values(by="F1", ascending=False)

    # Mostrar top 10
    print("🔝 TOP 10 combinaciones por F1-score:")
    display(df_global.head(10).round(4))

    # Gráfico de barras
    plt.figure(figsize=(14, 8))
    sns.barplot(
        data=df_global.head(20),  # mostrar top 20
        y="Etiqueta",
        x="F1",
        hue="Dataset",
        dodge=False
    )
    plt.title("Top 20 combinaciones por F1-score")
    plt.xlabel("F1-score")
    plt.ylabel("Método + Modelo")
    plt.tight_layout()
    plt.legend(title="Dataset")
    plt.show()


In [ ]:
# ================================
# CELDA 3: Heatmaps de F1, AUC Test y PR AUC por Imputación × Modelo
# ================================

if df_global.empty:
    print("⚠️ No se han encontrado datos para graficar.")
else:
    # Métricas que queremos visualizar
    metricas = [
        ("F1", "YlGnBu", "F1-score"),
        ("AUC_Test", "Blues", "AUC Test"),
        ("PR_AUC", "Oranges", "PR AUC")
    ]

    for metrica, cmap, titulo in metricas:
        plt.figure(figsize=(10, 6))
        tabla = df_global.pivot_table(
            index="Imputación",
            columns="Modelo",
            values=metrica,
            aggfunc="max"
        )
        sns.heatmap(tabla, annot=True, fmt=".3f", cmap=cmap, cbar_kws={"label": metrica})
        plt.title(f"{titulo} máximo por método de imputación y modelo")
        plt.xlabel("Modelo")
        plt.ylabel("Método de imputación")
        plt.tight_layout()
        plt.show()


In [ ]:
# ================================
# CELDA 4: Comparativa por tipo de faltantes y método de imputación
# ================================

if df_global.empty:
    print("⚠️ No hay datos suficientes para analizar tipo/porcentaje de faltantes.")
else:
    # Extraer tipo y porcentaje de faltantes del nombre del dataset
    def extraer_info_dataset(nombre):
        match = re.match(r"df_?([a-z]+)_(\d+)", nombre.lower())
        if match:
            tipo, porcentaje = match.groups()
            return tipo.upper(), int(porcentaje)
        else:
            return "ORIGINAL", 0  # dataset sin simulación de faltantes

    # Añadir columnas al DataFrame
    df_global["Tipo_Faltantes"], df_global["Porcentaje_Faltantes"] = zip(*df_global["Dataset"].map(extraer_info_dataset))

    # Mostrar tabla resumen
    resumen_datasets = df_global[["Dataset", "Tipo_Faltantes", "Porcentaje_Faltantes"]].drop_duplicates()
    display(resumen_datasets.sort_values(by=["Tipo_Faltantes", "Porcentaje_Faltantes"]))

    # === Versión A: Barras agrupadas por tipo de faltantes (ignorando porcentaje si es único) ===
    resumen_tipo = df_global.groupby(["Tipo_Faltantes", "Imputación"])["F1"].mean().reset_index()

    plt.figure(figsize=(10, 6))
    sns.barplot(data=resumen_tipo, x="Imputación", y="F1", hue="Tipo_Faltantes")
    plt.title("F1-score medio por tipo de faltantes y método de imputación")
    plt.ylabel("F1 medio")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # === Nueva visualización: Evolución por tipo de faltantes en tres subgráficos ===
    n_porcentajes = df_global["Porcentaje_Faltantes"].nunique()
    tipos_faltantes = ["MAR", "MCAR", "MNAR"]

    if n_porcentajes > 1 and all(t in df_global["Tipo_Faltantes"].unique() for t in tipos_faltantes):
        resumen_pct = df_global.groupby(["Tipo_Faltantes", "Porcentaje_Faltantes", "Imputación"])["F1"].mean().reset_index()

        fig, axs = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

        for i, tipo in enumerate(tipos_faltantes):
            df_tipo = resumen_pct[resumen_pct["Tipo_Faltantes"] == tipo]
            sns.lineplot(
                data=df_tipo,
                x="Porcentaje_Faltantes",
                y="F1",
                hue="Imputación",
                marker="o",
                ax=axs[i]
            )
            axs[i].set_title(f"{tipo}")
            axs[i].set_xlabel("% faltantes")
            axs[i].set_ylabel("F1 medio" if i == 0 else "")
            axs[i].legend().set_title("Imputación")
            axs[i].set_ylim(0.25, resumen_pct["F1"].max() + 0.05)

        plt.suptitle("Evolución del F1 medio por tipo de faltantes simulados (10-30%)", fontsize=14)
        plt.tight_layout()
        plt.show()

    else:
        print("ℹ️ No se han detectado múltiples porcentajes o faltan tipos de faltantes esperados.")


In [ ]:
# ================================
# CELDA 5: Comparativa agregada por modelo y método de imputación
# ================================

if df_global.empty:
    print("⚠️ No hay datos disponibles para análisis agregado.")
else:
    # Calcular medias y desviaciones por imputador y modelo
    resumen_agregado = df_global.groupby(["Imputación", "Modelo"]).agg(
        F1_Mean=("F1", "mean"),
        F1_Std=("F1", "std"),
        AUC_Test_Mean=("AUC_Test", "mean"),
        PR_AUC_Mean=("PR_AUC", "mean")
    ).reset_index()

    # Mostrar tabla resumen
    print("📋 Rendimiento medio por imputador y modelo:")
    display(resumen_agregado.sort_values(by="F1_Mean", ascending=False).round(4))

    # Gráfico: F1 medio con barras de error
    plt.figure(figsize=(12, 6))
    sns.barplot(
        data=resumen_agregado,
        x="Imputación",
        y="F1_Mean",
        hue="Modelo",
        capsize=0.1,
        err_kws={'linewidth': 1},
        palette="Set2"
    )
    plt.title("F1-score medio por imputación y modelo")
    plt.ylabel("F1 medio")
    plt.xlabel("Método de imputación")
    plt.tight_layout()
    plt.show()


In [ ]:
# ================================
# CELDA FINAL: Exportar todos los resultados globales a Parquet
# ================================

if df_global.empty:
    print("⚠️ No hay datos que exportar.")
else:
    output_file = "df_global_resultados.parquet"
    df_global.to_parquet(output_file, index=False)
    print(f"✅ Archivo guardado correctamente como: {output_file}")


In [ ]:
# ================================
# ANÁLISIS 1: Ranking global por modelo
# ================================

if df_global.empty:
    print("⚠️ No hay datos cargados para generar el ranking.")
else:
    # Agrupar por modelo e imputación, y calcular medias de cada métrica
    ranking_modelo = df_global.groupby(["Modelo", "Imputación"]).agg(
        F1_Mean=("F1", "mean"),
        F1_Std=("F1", "std"),
        AUC_Mean=("AUC_Test", "mean"),
        PR_AUC_Mean=("PR_AUC", "mean")
    ).reset_index()

    # Mostrar ranking ordenado por F1 medio dentro de cada modelo
    display(ranking_modelo.sort_values(by=["Modelo", "F1_Mean"], ascending=[True, False]).round(4))

    # Gráfico de barras por modelo con F1 medio
    plt.figure(figsize=(12, 6))
    sns.barplot(
        data=ranking_modelo,
        x="Imputación",
        y="F1_Mean",
        hue="Modelo",
        palette="Set1"
    )
    plt.title("Ranking de métodos de imputación por modelo (F1-score medio)")
    plt.ylabel("F1 medio")
    plt.xlabel("Método de imputación")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:
# ================================
# ANÁLISIS 2: Robustez por método de imputación
# ================================

if df_global.empty:
    print("⚠️ No hay datos cargados para analizar la robustez.")
else:
    # Gráfico 1: Boxplot de F1 por método de imputación (todas las combinaciones)
    plt.figure(figsize=(10, 6))
    sns.boxplot(
        data=df_global,
        x="Imputación",
        y="F1",
        hue="Imputación",
        dodge=False,
        legend=False
    )
    plt.title("Distribución de F1-score por método de imputación (robustez)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # Tabla: Rango y desviación de F1 por imputador
    stats_robustez = df_global.groupby("Imputación")["F1"].agg(
        F1_Mean="mean",
        F1_Std="std",
        F1_Min="min",
        F1_Max="max",
        F1_Range=lambda x: x.max() - x.min()
    ).sort_values(by="F1_Range", ascending=True)

    print("📋 Estadísticas de robustez por método de imputación (F1):")
    display(stats_robustez.round(4))


In [ ]:
# ================================
# ANÁLISIS 3: Comparación por tipo de faltantes (MCAR, MAR, MNAR)
# ================================

if df_global.empty:
    print("⚠️ No hay datos disponibles para analizar tipos de faltantes.")
else:
    # Asegurar que las columnas Tipo_Faltantes y Porcentaje_Faltantes existen
    if "Tipo_Faltantes" not in df_global.columns or "Porcentaje_Faltantes" not in df_global.columns:
        def extraer_info_dataset(nombre):
            match = re.match(r"df_?([a-z]+)_(\d+)", nombre.lower())
            if match:
                tipo, porcentaje = match.groups()
                return tipo.upper(), int(porcentaje)
            else:
                return "ORIGINAL", 0

        df_global["Tipo_Faltantes"], df_global["Porcentaje_Faltantes"] = zip(*df_global["Dataset"].map(extraer_info_dataset))

    # Filtrar los que sí tienen tipo simulado (excluye ORIGINAL)
    df_simulados = df_global[df_global["Tipo_Faltantes"] != "ORIGINAL"]

    if df_simulados.empty:
        print("ℹ️ No se han encontrado datasets con tipo de faltantes simulado (MCAR, MAR, MNAR).")
    else:
        # Gráfico de barras: F1 medio por tipo de faltantes y método de imputación
        resumen_tipo = df_simulados.groupby(["Tipo_Faltantes", "Imputación"])["F1"].mean().reset_index()

        plt.figure(figsize=(10, 6))
        sns.barplot(data=resumen_tipo, x="Imputación", y="F1", hue="Tipo_Faltantes")
        plt.title("F1-score medio por tipo de faltantes y método de imputación")
        plt.ylabel("F1 medio")
        plt.xlabel("Método de imputación")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.legend(title="Tipo de faltantes", bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.show()


In [ ]:
# ================================
# ANÁLISIS 4: F1-score vs porcentaje de faltantes simulados
# ================================

if df_global.empty:
    print("⚠️ No hay datos cargados para analizar el efecto del porcentaje de faltantes.")
else:
    # Asegurar columnas necesarias
    if "Tipo_Faltantes" not in df_global.columns or "Porcentaje_Faltantes" not in df_global.columns:
        def extraer_info_dataset(nombre):
            match = re.match(r"df_?([a-z]+)_(\d+)", nombre.lower())
            if match:
                tipo, porcentaje = match.groups()
                return tipo.upper(), int(porcentaje)
            else:
                return "ORIGINAL", 0

        df_global["Tipo_Faltantes"], df_global["Porcentaje_Faltantes"] = zip(*df_global["Dataset"].map(extraer_info_dataset))

    # Filtrar simulaciones
    df_simulados = df_global[df_global["Tipo_Faltantes"] != "ORIGINAL"]

    if df_simulados["Porcentaje_Faltantes"].nunique() <= 1:
        print("ℹ️ Solo se ha detectado un único porcentaje de faltantes. No se puede generar el gráfico de evolución.")
    else:
        # F1 medio por imputador y porcentaje de faltantes
        resumen_pct = df_simulados.groupby(["Porcentaje_Faltantes", "Imputación"])["F1"].mean().reset_index()

        plt.figure(figsize=(12, 6))
        sns.lineplot(
            data=resumen_pct,
            x="Porcentaje_Faltantes",
            y="F1",
            hue="Imputación",
            marker="o"
        )
        plt.title("Evolución del F1 medio según porcentaje de faltantes por método de imputación")
        plt.ylabel("F1-score medio")
        plt.xlabel("Porcentaje de faltantes simulados")
        plt.tight_layout()
        plt.legend(title="Método de imputación", bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.show()


        # === Gráfico adicional: Evolución F1 por % faltantes y tipo de mecanismo (subplots por tipo) ===
        mecanismos = ["MCAR", "MAR", "MNAR"]
        resumen_mecanismo = df_global[df_global["Tipo_Faltantes"].isin(mecanismos)].groupby(
            ["Tipo_Faltantes", "Porcentaje_Faltantes", "Imputación"]
        )["F1"].mean().reset_index()

        fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

        for i, tipo in enumerate(mecanismos):
            ax = axes[i]
            subset = resumen_mecanismo[resumen_mecanismo["Tipo_Faltantes"] == tipo]
            sns.lineplot(
                data=subset,
                x="Porcentaje_Faltantes",
                y="F1",
                hue="Imputación",
                marker="o",
                ax=ax
            )
            ax.set_title(f"{tipo}")
            ax.set_xlabel("% faltantes")
            ax.set_ylabel("F1 medio" if i == 0 else "")
            ax.legend().set_title("Imputación")
            ax.set_ylim(None, 0.5)

        plt.suptitle("Evolución del F1 medio por tipo de faltantes simulados (10-30%)", y=1.02)
        plt.tight_layout()
        plt.show()


In [ ]:
# ================================
# ANÁLISIS 5: Correlación entre métricas (F1, AUC, PR AUC)
# ================================

if df_global.empty:
    print("⚠️ No hay datos cargados para analizar correlaciones.")
else:
    # Seleccionar solo las métricas numéricas
    corr_df = df_global[["F1", "AUC_Test", "PR_AUC"]]

    # Matriz de correlación
    matriz_corr = corr_df.corr()

    print("🔗 Matriz de correlación entre métricas:")
    display(matriz_corr.round(3))

    # Heatmap de correlación
    plt.figure(figsize=(6, 4))
    sns.heatmap(
        matriz_corr,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
        cbar_kws={"label": "Correlación"}
    )
    plt.title("Correlación entre métricas de evaluación")
    plt.tight_layout()
    plt.show()


In [ ]:
# ================================
# ANÁLISIS 6: Radar plot por método de imputación
# ================================

from math import pi

if df_global.empty:
    print("⚠️ No hay datos disponibles para el radar plot.")
else:
    # Promediar métricas por método de imputación
    radar_df = df_global.groupby("Imputación")[["F1", "AUC_Test", "PR_AUC"]].mean().reset_index()

    # Normalizar cada métrica entre 0 y 1 (opcional para visualizar mejor)
    radar_norm = radar_df.copy()
    radar_norm[["F1", "AUC_Test", "PR_AUC"]] = radar_norm[["F1", "AUC_Test", "PR_AUC"]].apply(
        lambda x: (x - x.min()) / (x.max() - x.min())
    )

    # Preparar datos para radar plot
    categorias = ["F1", "AUC_Test", "PR_AUC"]
    N = len(categorias)

    plt.figure(figsize=(8, 8))
    for i, row in radar_norm.iterrows():
        valores = row[categorias].tolist()
        valores += valores[:1]  # cerrar el círculo
        angulos = [n / float(N) * 2 * pi for n in range(N)]
        angulos += angulos[:1]
        plt.polar(angulos, valores, label=row["Imputación"])
        plt.fill(angulos, valores, alpha=0.1)

    plt.xticks([n / float(N) * 2 * pi for n in range(N)], categorias)
    plt.yticks([0.25, 0.5, 0.75, 1.0], ["0.25", "0.5", "0.75", "1.0"])
    plt.title("Radar plot de métricas normalizadas por método de imputación")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()


In [ ]:
# Crear DataFrame de métricas medias por modelo
df_modelo_metrics = df_global.groupby("Modelo")[["F1", "AUC_Test", "PR_AUC"]].mean().reset_index()

# Normalizar métricas a [0, 1]
df_norm = df_modelo_metrics.copy()
cols_metrics = ["F1", "AUC_Test", "PR_AUC"]
df_norm[cols_metrics] = (df_norm[cols_metrics] - df_norm[cols_metrics].min()) / (df_norm[cols_metrics].max() - df_norm[cols_metrics].min())

# Coordenadas del radar plot
labels = cols_metrics
num_vars = len(labels)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]  # Cierre del radar

# Crear la figura
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
for i, row in df_norm.iterrows():
    valores = row[cols_metrics].tolist()
    valores += valores[:1]
    ax.plot(angles, valores, label=row["Modelo"])
    ax.fill(angles, valores, alpha=0.1)

# Ajustes estéticos
ax.set_title("Radar plot de métricas normalizadas por modelo", y=1.08)
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_thetagrids(np.degrees(angles[:-1]), labels)
ax.set_ylim(0, 1)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

df_metricas_modelo_imputacion = (
    df_global.groupby(["Imputación", "Modelo"])
    .agg(F1_Mean=("F1","mean"), AUC_Test_Mean=("AUC_Test","mean"), PR_AUC_Mean=("PR_AUC","mean"))
    .reset_index()
)

df_radar = df_metricas_modelo_imputacion.copy()
df_radar["Combinación"] = df_radar["Imputación"] + " + " + df_radar["Modelo"]

labels = ["F1", "AUC_Test", "PR_AUC"]
angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist() + [0]

scaled = pd.DataFrame(
    MinMaxScaler().fit_transform(df_radar[["F1_Mean","AUC_Test_Mean","PR_AUC_Mean"]]),
    columns=labels
)

fig = plt.figure(figsize=(18, 11))
ax = fig.add_axes([0.0, 0.0, 0.62, 1.0], polar=True)

colors = [plt.cm.tab20(i % 20) for i in range(len(df_radar))]
lstyles = ['-','--','-.',':']

for i, (_, row) in enumerate(scaled.iterrows()):
    vals = row[labels].tolist() + [row[labels[0]]]
    ax.plot(angles, vals, color=colors[i], linestyle=lstyles[i%4], linewidth=1.8,
            label=df_radar["Combinación"].iloc[i])
    ax.fill(angles, vals, color=colors[i], alpha=0.04)

ax.set_theta_offset(np.pi/2)
ax.set_theta_direction(-1)
ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=14)
ax.set_ylim(0, 1)
ax.set_title("Radar plot métricas por modelo + imputación", y=1.08, fontsize=20, fontweight='bold')

handles, lbls = ax.get_legend_handles_labels()
fig.legend(handles, lbls, loc='center left', bbox_to_anchor=(0.63, 0.5),
           fontsize=12, ncol=1, framealpha=0.9,
           title="Combinación", title_fontsize=13,
           handlelength=2.5, labelspacing=0.8)

plt.show()

In [ ]:
# ================================
# ANÁLISIS 7: Tabla estilo artículo (mejor resultado por imputador y modelo)
# ================================

if df_global.empty:
    print("⚠️ No hay datos disponibles para generar tabla de resumen.")
else:
    # Seleccionar el mejor resultado por combinación de Imputación + Modelo (según F1)
    best_table = (
        df_global.sort_values(by="F1", ascending=False)
        .groupby(["Imputación", "Modelo"])
        .first()
        .reset_index()
    )

    # Seleccionar columnas clave
    columnas = ["Imputación", "Modelo", "F1", "AUC_Test", "PR_AUC"]
    tabla_export = best_table[columnas].sort_values(by=["Modelo", "F1"], ascending=[True, False])

    # Mostrar como tabla
    print("📋 Tabla estilo artículo (mejor resultado por modelo e imputador):")
    display(tabla_export.round(4))

    # Exportar como CSV y LaTeX si se desea
    tabla_export.to_csv("tabla_resultados_resumen.csv", index=False)
    with open("tabla_resultados_resumen.tex", "w") as f:
        f.write(tabla_export.round(4).to_latex(index=False))

    print("✅ Tabla exportada como:")
    print(" - 📄 CSV: tabla_resultados_resumen.csv")
    print(" - 📄 LaTeX: tabla_resultados_resumen.tex")
